<a href="https://colab.research.google.com/github/elkins/synth-saxs/blob/main/examples/interactive_tutorials/hydration_shell_analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Interactive Hydration Shell Analysis

In Small-Angle X-ray Scattering (SAXS), the protein is not a "dry" object in a vacuum. It is surrounded by solvent (water). One of the most subtle but important effects in SAXS modeling is the **hydration shell**—a layer of water molecules immediately surrounding the protein that is slightly denser (~2-5%) than bulk water.

This tutorial demonstrates how this "excess" density affects the scattering profile and the perceived size of the protein.

In [ ]:
import sys

# Install dependencies if running in Colab or a new environment
if "google.colab" in sys.modules:
    %pip install -q synth-saxs biotite matplotlib ipywidgets
else:
    sys.path.append("../../")

In [ ]:
import biotite.database.rcsb as rcsb
import biotite.structure.io as strucio
import matplotlib.pyplot as plt
from ipywidgets import FloatSlider, interact

from synth_saxs import calculate_radius_of_gyration, calculate_saxs_profile

print("Setup complete.")

## 1. Load a Reference Structure
We will use **Ubiquitin (1UBQ)**, a small and well-studied protein.

In [ ]:
# Download and load 1UBQ
pdb_file = rcsb.fetch("1UBQ", "pdb", ".")
structure = strucio.load_structure(pdb_file)
if hasattr(structure, "stack_depth") and structure.stack_depth() > 1:
    structure = structure[0]

# Pre-calculate the Radius of Gyration (dry)
rg_dry = calculate_radius_of_gyration(structure)
print(f"Structure loaded. Dry Rg: {rg_dry:.2f} A")

## 2. Interactive Visualization

The hydration shell is modeled by adding an excess density $\Delta\rho_{shell}$ to the solvent term. 

The effective scattering factor $f_{eff}(q)$ for an atom is roughly:
$$ f_{eff}(q) = f_{vac}(q) - (\rho_{sol} - \Delta\rho_{shell}) \cdot V \cdot \exp(-q^2 R^2 / 10) $$

As $\Delta\rho_{shell}$ increases, the "contrast" between the protein and its immediate surroundings changes, making the protein appear larger or "brighter" to X-rays.

In [ ]:
# The bulk profile does not change with the slider, so compute it once.
q_bulk, i_bulk = calculate_saxs_profile(structure, hydration_shell_density=0.0, n_points=100)


def update_plot(shell_density):
    # 1. Calculate profiles
    q = q_bulk
    _, i_shell = calculate_saxs_profile(
        structure, hydration_shell_density=shell_density, n_points=100
    )

    # 2. Plotting
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

    # Log-linear plot
    ax1.semilogy(q, i_bulk, "k--", label="Bulk Solvent only", alpha=0.5)
    ax1.semilogy(q, i_shell, "r-", linewidth=2, label=f"Shell Density: {shell_density:.3f} e/A^3")
    ax1.set_xlabel("q (A^-1)", fontsize=12)
    ax1.set_ylabel("Intensity (log)", fontsize=12)
    ax1.set_title("SAXS Profile I(q)")
    ax1.legend()
    ax1.grid(True, alpha=0.3)

    # Kratky plot (q^2 * I(q) vs q)
    ax2.plot(q, (q**2) * i_bulk, "k--", label="Bulk Solvent only", alpha=0.5)
    ax2.plot(q, (q**2) * i_shell, "r-", linewidth=2, label=f"Shell Density: {shell_density:.3f}")
    ax2.set_xlabel("q (A^-1)", fontsize=12)
    ax2.set_ylabel("q^2 * I(q)", fontsize=12)
    ax2.set_title("Kratky Plot")
    ax2.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.show()


# Create an interactive slider
interact(
    update_plot,
    shell_density=FloatSlider(
        value=0.03, min=0.0, max=0.1, step=0.01, description="Density (e/A^3):"
    ),
);